In [1]:
#imports

#from tensorzinb.tensorzinb import TensorZINB
import scMPRAforge as scm
from scMPRAforge.core import _smart_matrix, _mom_from_training_data, _matricies_to_order, _tensorzinb_fit

2026-01-22 15:52:42.157561: I tensorflow/core/util/util.cc:169] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


In [2]:
import pandas as pd
import numpy as np
import time
import pickle
from formulaic import Formula
import seaborn as sns
import matplotlib.pyplot as plt

In [3]:
#create dask cluster

from dask_jobqueue import SLURMCluster
from dask.distributed import Client

cluster=SLURMCluster(
    cores=8,#cores per slurm job
    memory="512G",#memory per slurm job
    processes=1,#dask workers per slurm job
    job_extra_directives=["-p day", 
        f"--job-name=simclust_worker",
        f"--time=6:00:00",
        f"--output=worker_%j.out"]
)

cluster.scale(jobs=1)

client = Client(cluster,
        timeout=f"{5*60}s",   # Client <-> scheduler timeout 
        heartbeat_interval="20s"  # Worker heartbeat interval
    )

#from dask.distributed import Client, LocalCluster
#cluster=LocalCluster(memory_limit='8GB')
#client = Client(cluster)

In [4]:
print(client.dashboard_link)

http://10.18.22.42:8787/status


# Describe with Ortho


In [5]:
data_root="/nfs/roberts/project/pi_skr2/shared/tabula_data"

def load_and_preprocess_data(data_root):
    """Load, preprocess the data and return processed data object."""
    seelig = scm.scMPRA_data.from_tsv(f"{data_root}/seelig/seelig_counts_grouped.txt")
    seelig.set_negative_controls(["AACGCCCTCCACGGATGGGCCGGCCAATAAGAAGCGTTAGCGGACTCATGCGTTACGCGCCTCCGAGTTATGGGGGGGGAGGCGCGTATCTCGTGGAGAAGAAGCGATGTAACGCTTGGGCGATAAGCTTATAAGGAAGATATTT",
    "CCCTCGGAGTTAATAAGATACGCGGATCGATATCGGCTTGAAGAAGCGTATCTTATCTTCAGATGGGGATGTCGCGCATCCACCCAGTGGGCACCGCCGCTATAGAAGGGTGATAACGCTTCTCAGCCTTCAGGCTCTGGGTCTT"])
    seelig.set_reference_cell("HEPG2")
    seelig.ortho_filter()
    return seelig



seelig = load_and_preprocess_data(data_root)

scMPRAforge: INFO: Dropped 81 of 2688 (cell_type, cre_id) combos with fewer than 3 nonzero entries.


In [6]:
def single_model_fit(data, split):
    data = data.data
    levels=data[split].unique()
    t = levels[0]

    t_future = client.submit(
            _smart_matrix,
            data=data[data[split]==t],
            split=split
        )

    if split=="cell_type":
        init_method="pass"
        init_vals=client.submit(_mom_from_training_data, 
                data=data,
                split="cell_type",
                subset=t,
                indicies=client.submit(_matricies_to_order, matricies=t_future)
                )

    else:
            init_method="nb"
            init_vals= None 

    tzinb_futures = client.submit(
                _tensorzinb_fit,
                t_future,
                t,
                init_method=init_method,
                init_vals=init_vals
            )

    return(scm.experiment_model(model=tzinb_futures,
                                split=split),
            t_future)



In [7]:
f = single_model_fit(seelig, 'cell_type')


In [8]:
print("finished!")

finished!


In [9]:
# client.close()
# cluster.close()